In [7]:
import os
import re
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# ------------------------- 配置 -------------------------
IMAGE_FOLDER = r"E:\YRH-HSI\python data\MUST\must\HSI-TIFF_GAM"
OUTPUT_DIR = os.path.join(IMAGE_FOLDER, "GradCAM_Results")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------- 图像预处理 -------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ------------------------- 加载模型 -------------------------
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model = model.to(DEVICE)
model.eval()

# 获取最后一个卷积层的输出（用于 Grad‑CAM）
final_conv_layer = model.layer4[-1].conv3  # ResNet50 的 layer4 最后一个 bottleneck 的 conv3

# 注册前向钩子以捕获特征图
feature_maps = []
def forward_hook(module, input, output):
    feature_maps.append(output)
forward_handle = final_conv_layer.register_forward_hook(forward_hook)

# ------------------------- Grad‑CAM 函数 -------------------------
def grad_cam(image_tensor, target_class=None):
    """
    输入: image_tensor (1,3,224,224) 已归一化
    返回: 热力图 (224,224) numpy 数组
    """
    global feature_maps
    feature_maps.clear()  # 清空之前的特征图

    image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
    image_tensor.requires_grad_(True)

    # 前向传播
    output = model(image_tensor)
    if target_class is None:
        target_class = output.argmax(dim=1).item()

    # 计算目标类别的梯度
    model.zero_grad()
    class_score = output[0, target_class]
    class_score.backward()

    # 获取特征图和梯度
    features = feature_maps[0]  # shape: [1, C, H, W]
    grads = image_tensor.grad  # 这是对输入图像的梯度，但我们想要对特征图的梯度
    # 实际上，我们需要对特征图的梯度，但 backward 后梯度保存在 feature_maps 的 grad_fn 里？
    # 标准 Grad‑CAM 需要计算特征图相对于类别得分的梯度，而非输入图像的梯度。
    # 正确做法：重新前向并保存特征图，然后反向传播得到特征图的梯度。
    # 更方便：使用 register_backward_hook 或重新实现。
    # 以下采用重新计算的方式：
    
    # 清理梯度，重新前向
    model.zero_grad()
    feature_maps.clear()
    output = model(image_tensor)
    class_score = output[0, target_class]
    
    # 反向传播，保留中间梯度
    class_score.backward(retain_graph=True)
    
    # 获取特征图的梯度（梯度保存在对应张量的 grad 属性中）
    # 我们通过钩子保存特征图，但没有保存梯度，所以需要获取特征图的 grad
    # 更稳健的方法：使用 hooks 捕获特征图和其梯度
    # 改用下面的实现（使用 register_full_backward_hook 或简单方法）。
    
    # 简化：使用 torch.autograd.grad 直接计算特征图相对于类得分的梯度
    # 但需要特征图作为输出，这里我们直接取 hook 保存的特征图
    # 由于我们无法直接获取特征图的梯度，下面采用经典实现：
    
    # 重新实现：使用特定 hooks 同时捕获 forward 和 backward
    # 这里为简洁，我们采用另一种常见方式：用 torchvision 的 GradCAM 包，但为了无额外依赖，我们手动实现。
    
    # 这里为了代码完整性，我们修正实现：
    # 改用自包含的 Grad‑CAM 类，从零实现，确保正确。
    # 鉴于篇幅，我将在下面给出标准实现，替换当前函数。
    pass

# ------------------------- 标准 Grad‑CAM 实现（修正版） -------------------------
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def __call__(self, input_tensor, target_class=None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        class_score = output[0, target_class]
        class_score.backward()
        
        # 获取梯度和激活
        grads = self.gradients  # [1, C, H, W]
        acts = self.activations # [1, C, H, W]
        
        # 全局平均池化梯度 -> 每个通道的权重
        weights = torch.mean(grads, dim=(2, 3), keepdim=True)  # [1, C, 1, 1]
        # 加权求和
        cam = torch.sum(weights * acts, dim=1, keepdim=True)   # [1, 1, H, W]
        cam = torch.relu(cam)   # ReLU 只保留正贡献
        # 归一化到 [0,1]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy()

# 创建 GradCAM 对象，指定最后一个卷积层
grad_cam = GradCAM(model, final_conv_layer)

# ------------------------- 处理所有图片 -------------------------
image_paths = [
    os.path.join(IMAGE_FOLDER, f)
    for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".tiff", ".tif"))
]

# 按文件名排序（可选）
def sort_key(path):
    base = os.path.basename(path)
    nums = re.findall(r'\d+', base)
    return int(nums[0]) if nums else 0
image_paths.sort(key=sort_key)

for img_path in tqdm(image_paths, desc="Generating Grad‑CAM"):
    # 加载原图
    img_pil = Image.open(img_path).convert("RGB")
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)
    
    # 计算热力图
    cam = grad_cam(img_tensor)  # shape (H, W)
    
    # 将热力图缩放到原图尺寸（原图可能 resize 过，但我们用 224x224 的）
    # 为了可视化，我们 resize 到 224x224（与输入一致）
    cam_resized = Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
    cam_resized = np.array(cam_resized) / 255.0
    
    # 叠加热力图到原图（也需要 resize 到 224x224 以匹配）
    img_resized = img_pil.resize((224, 224))
    img_np = np.array(img_resized) / 255.0
    
    # 生成热力图颜色映射
    heatmap = plt.cm.jet(cam_resized)[:, :, :3]  # 取 RGB
    superimposed = heatmap * 0.5 + img_np * 0.5
    superimposed = np.clip(superimposed, 0, 1)
    
    # 保存
    base_name = os.path.basename(img_path)
    save_name = f"GradCAM_{base_name}.png"
    save_path = os.path.join(OUTPUT_DIR, save_name)
    
    plt.imsave(save_path, superimposed)
    
    # 可选：同时保存热力图单独
    # plt.imsave(os.path.join(OUTPUT_DIR, f"heatmap_{base_name}.png"), cam_resized, cmap='jet')

print(f"完成！所有可视化结果已保存至: {OUTPUT_DIR}")

Generating Grad‑CAM: 100%|███████████████████████████████████████████████████████████| 148/148 [00:03<00:00, 44.82it/s]

完成！所有可视化结果已保存至: E:\YRH-HSI\python data\MUST\must\HSI-TIFF_GAM\GradCAM_Results


In [1]:
import os
import re
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

IMAGE_FOLDER = "HSI-TIFF_GAM"
BASE_OUTPUT_DIR = os.path.join(IMAGE_FOLDER, "GradCAM_Results_Enhanced")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ENHANCED_FOLDERS = [
    "01_strong_gamma",
    "02_power2",
    "03_percentile_thresh",
    "04_histogram_equalize"
]

for folder in ENHANCED_FOLDERS:
    os.makedirs(os.path.join(BASE_OUTPUT_DIR, folder), exist_ok=True)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model = model.to(DEVICE)
model.eval()

final_conv_layer = model.layer4[-1].conv3

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, input_tensor, target_class=None):
        self.model.zero_grad()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        class_score = output[0, target_class]
        class_score.backward()

        grads = self.gradients
        acts = self.activations

        weights = torch.mean(grads, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * acts, dim=1, keepdim=True)
        cam = torch.relu(cam)

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy()

grad_cam = GradCAM(model, final_conv_layer)

def histogram_equalize(cam):
    img_uint8 = (cam * 255).astype(np.uint8)
    hist, bins = np.histogram(img_uint8.flatten(), 256, [0, 256])
    cdf = hist.cumsum()
    cdf_normalized = cdf / cdf[-1]
    equalized = np.interp(img_uint8.flatten(), bins[:-1], cdf_normalized * 255)
    return equalized.reshape(img_uint8.shape) / 255.0

configs = [
    {
        "name": "strong_gamma",
        "cmap": "inferno",
        "alpha": 0.6,
        "postprocess": lambda cam: (lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))(np.power(cam, 0.3))
    },
    {
        "name": "power2",
        "cmap": "inferno",
        "alpha": 0.6,
        "postprocess": lambda cam: np.power(cam, 2)
    },
    {
        "name": "percentile_thresh",
        "cmap": "inferno",
        "alpha": 0.6,
        "postprocess": lambda cam: (lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8))(cam * (cam >= np.percentile(cam, 80)))
    },
    {
        "name": "histogram_equalize",
        "cmap": "inferno",
        "alpha": 0.6,
        "postprocess": histogram_equalize
    }
]

image_paths = [
    os.path.join(IMAGE_FOLDER, f)
    for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".tiff", ".tif"))
]

def sort_key(path):
    base = os.path.basename(path)
    nums = re.findall(r'\d+', base)
    return int(nums[0]) if nums else 0
image_paths.sort(key=sort_key)

for img_path in tqdm(image_paths, desc="Generating Enhanced Grad-CAM"):
    img_pil = Image.open(img_path).convert("RGB")
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

    cam_raw = grad_cam(img_tensor)

    img_resized = img_pil.resize((224, 224))
    img_np = np.array(img_resized) / 255.0

    base_name = os.path.basename(img_path)

    for cfg, folder in zip(configs, ENHANCED_FOLDERS):
        cam = cfg["postprocess"](cam_raw.copy())
        cam = np.clip(cam, 0, 1)

        cam_resized = np.array(
            Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
        ) / 255.0

        heatmap = plt.get_cmap(cfg["cmap"])(cam_resized)[:, :, :3]

        alpha = cfg["alpha"]
        superimposed = heatmap * alpha + img_np * (1 - alpha)
        superimposed = np.clip(superimposed, 0, 1)

        save_name = f"GradCAM_{os.path.splitext(base_name)[0]}.png"
        save_path = os.path.join(BASE_OUTPUT_DIR, folder, save_name)
        plt.imsave(save_path, superimposed)

print("Done! Enhanced results saved to:")
for folder in ENHANCED_FOLDERS:
    print(f"  {os.path.join(BASE_OUTPUT_DIR, folder)}")

Generating Enhanced Grad-CAM: 100%|██████████████████████████████████████████████████| 148/148 [00:11<00:00, 12.82it/s]

Done! Enhanced results saved to:
  HSI-TIFF_GAM\GradCAM_Results_Enhanced\01_strong_gamma
  HSI-TIFF_GAM\GradCAM_Results_Enhanced\02_power2
  HSI-TIFF_GAM\GradCAM_Results_Enhanced\03_percentile_thresh
  HSI-TIFF_GAM\GradCAM_Results_Enhanced\04_histogram_equalize
